In [2]:
import math
import pathlib
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

# Find the repo root, so `from jabarti_llm...` works wherever this is opened.
ROOT = pathlib.Path.cwd()
while not (ROOT / "jabarti_llm" / "config.py").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

torch.manual_seed(0)

print("torch     :", torch.__version__)
print("repo root :", ROOT)

torch     : 2.15.0+cpu
repo root : d:\COURSES\jabarti_llm


In [2]:
from jabarti_llm.config import ModelConfig

TOKENS = ["the", "cat", "sat", "on", "mat"]

cfg = ModelConfig(
    vocab_size=32,
    d_model=8,
    n_heads=2,
    n_layers=2,
    max_seq_len=16,
    dropout=0.1,
)

B = 1
T = len(TOKENS)

x = torch.randn(B, T, cfg.d_model)

print("d_model    :", cfg.d_model)
print("n_heads    :", cfg.n_heads, " -> d_k =", cfg.d_k)
print("n_layers   :", cfg.n_layers)
print("d_ff       :", cfg.d_ff, " <- __post_init__ filled this in: 4 * d_model")
print("vocab_size :", cfg.vocab_size)
print("max_seq_len:", cfg.max_seq_len)

d_model    : 8
n_heads    : 2  -> d_k = 4
n_layers   : 2
d_ff       : 3072  <- __post_init__ filled this in: 4 * d_model
vocab_size : 32
max_seq_len: 16


In [3]:
from jabarti_llm.model.attention import MultiHeadAttention
from jabarti_llm.model.feedforward import FeedForward

attn = MultiHeadAttention(cfg)
ffn = FeedForward(cfg)

attn_out, updated_cache = attn(x)
y = ffn(attn_out)

print("x     :", tuple(x.shape))
print("attn  :", tuple(attn_out.shape))
print("ffn   :", tuple(y.shape))
print()

print("same shape in and out?", tuple(y.shape) == tuple(x.shape))

x     : (1, 5, 8)
attn  : (1, 5, 8)
ffn   : (1, 5, 8)

same shape in and out? True


In [4]:
ln = nn.LayerNorm(cfg.d_model)

wild_input = torch.tensor([
    [
        [50.0, 52.0, 48.0, 51.0, 49.0, 53.0, 47.0, 50.0]
    ]
])

normed_input = ln(wild_input)

print(wild_input.shape)
print(normed_input.shape)

torch.Size([1, 1, 8])
torch.Size([1, 1, 8])


In [5]:
print("wild_input: ")
print("  mean:", round(wild_input.mean().item(), 3),
      "  std:", round(wild_input.std(unbiased=False).item(), 3))
print()

print("normed_input: ")
print("  mean:", round(normed_input.mean().item(), 3),
      "  std:", round(normed_input.std(unbiased=False).item(), 3))
print()

wild_input: 
  mean: 50.0   std: 1.871

normed_input: 
  mean: 0.0   std: 1.0



In [6]:
from jabarti_llm.model.attention import MultiHeadAttention
from jabarti_llm.model.feedforward import FeedForward
from jabarti_llm.model.block import TransformerBlock

attn = MultiHeadAttention(cfg)
ffn = FeedForward(cfg)

attn_out, updated_cache = attn(x)
y = ffn(attn_out)

blocks = nn.ModuleList(
    TransformerBlock(cfg)
    for ix in range(cfg.n_layers)
)

In [7]:
ln_final = nn.LayerNorm(cfg.d_model)
lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

normed = ln_final(x)
logits = lm_head(normed)

In [8]:
TOKENS = ["the", "cat", "sat", "on", "mat"]

last = logits[0, -1]
probs = F.softmax(last, dim=-1)

top = torch.topk(probs, 5)

In [9]:
from jabarti_llm.model.embeddings import InputEmbedding

embedding = InputEmbedding(cfg)
head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

print("token_embedding.weight:", embedding.token_embedding.weight.shape)
print("head.weight:", head.weight.shape)

head.weight = embedding.token_embedding.weight

token_embedding.weight: torch.Size([32, 8])
head.weight: torch.Size([32, 8])


In [10]:
def _init_weights(module):
    """GPT-2's initialisation: normal(0, 0.02), zeroed biases.

        PyTorch's Linear default depends on fan-in, which leaves a 768-wide
        model's activations larger than GPT-2 assumes and makes the early
        steps of training less stable.
    """

    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weights, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weights, mean=0.0, std=0.02)

def _scale_residual_projections(blocks, config):

    scale = math.sqrt( 2 *  config.n_layers)
    for block in blocks:
        with torch.no_grad():
            block.attn.W_o.weight.div_(scale)
            block.ffn.fc2.weight.div_(scale)

In [11]:
sentence = torch.tensor([[5, 9, 14, 3, 21, 7]])

inputs = sentence[:, 0:-1]
targets = sentence[:, 1:]

print("sentence:", sentence.tolist()[0])
print("inputs  :", inputs.tolist()[0])
print("targets :", targets.tolist()[0])

print()
print(f"{'position':>9}{'reads':>8}{'must predict':>14}")
for t in range(inputs.size(1)):
    print(f"{t:>9}{inputs[0, t].item():>8}{targets[0, t].item():>14}")

sentence: [5, 9, 14, 3, 21, 7]
inputs  : [5, 9, 14, 3, 21]
targets : [9, 14, 3, 21, 7]

 position   reads  must predict
        0       5             9
        1       9            14
        2      14             3
        3       3            21
        4      21             7


In [12]:
ln_final = nn.LayerNorm(cfg.d_model)
lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

normed = ln_final(x)
logits = lm_head(normed)

print("logits          :", tuple(logits.shape)) # (B, T, vocab_size) 
print("targets          :", tuple(targets.shape)) # (B, T)

# flatten (B*T, vocab_size) and (B*T,) for loss computation

flatten_logits = logits.reshape(-1 , logits.size(-1))
print("flatten_logits          :", tuple(flatten_logits.shape))

flatten_targets = targets.reshape(-1)
print("flatten_targets          :", tuple(flatten_targets.shape))

# === Example:

# logits       (B, T, V)   = (2, 5, 50257)   ← scores for every token at every position
# targets      (B, T)      = (2, 5)          ← the correct token id at each position

# logits.reshape(-1, lg.size(-1))  →  (10, 50257)   ← 10 separate predictions
# targets.reshape(-1)              →  (10,)         ← 10 correct answers

# ===

loss = F.cross_entropy( flatten_logits, flatten_targets )

# loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

logits          : (1, 5, 32)
targets          : (1, 5)
flatten_logits          : (5, 32)
flatten_targets          : (5,)


In [15]:
# fill to 12
sentence = torch.tensor([[5, 9, 14, 3, 21, 7, cfg.pad_id, cfg.pad_id, cfg.pad_id,
                                              cfg.pad_id, cfg.pad_id, cfg.pad_id]])

loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                       ignore_index=cfg.pad_id)

AttributeError: 'ModelConfig' object has no attribute 'pad_id'

In [6]:
from jabarti_llm.config import ModelConfig
from jabarti_llm.model.gpt import GPT

my_gpt = GPT(ModelConfig.jabarti())
print("jabarti():")
print("total parameters : ", f"{my_gpt.num_parameters():,}")

jabarti():
total parameters :  50,512,896
